# 25 — RPMR Colab smoke test

Upload this notebook to **Google Colab**. Choose **GPU**, then **Run all**.
No repository clone, local Python setup, adapter upload, or training is needed.
Allow Google Drive and enter the current balance and **the live units/hour shown by Colab**.

**Remaining budget:** about 50 units. Keep 20 units. This smoke session is capped at
**five estimated units**, including installation and model download after the first cell starts.
Earlier allocated-runtime time is outside this notebook's clock; use the current balance.
At the historical A100 rate of 6.77/hour, five units allow about 44 minutes.
Planning estimate: 10–30 minutes (about 1.1–3.4 historical-rate units), not measured.
T4/L4/A100 rates and actual setup time differ. Do not increase the cap to bypass a failure.

**What:** test the new projection calculations and access to Qwen3.5-2B's real cache.
**Why:** verify compatibility and timing before a larger experiment.
**Input:** synthetic matrices and one hand-written code question.
**Output:** saved checks, environment, memory shapes, raw prefix, and decode timing.
**Design:** one model, one selected head, 16 greedy thinking tokens, three single-token
continuations. No model-written code is executed and no benchmark accuracy is measured.

The cache repair is deliberately forced for plumbing validation. Its candidate direction
comes from prompt-to-prefix memory drift; this is **not a detected loop** and is not the
final online recovery algorithm. Greedy decoding is diagnostic, not the thesis sampler.
No automatic pilot or full study follows this notebook.

Results are saved under `MyDrive/stop-overthinking/rpmr-smoke/<unique-run>/`.
The run cell saves a report on success or failure and tries to disconnect the GPU.
A wall-time watchdog also attempts disconnection before the estimated unit cap.
These use your entered rate, not live account billing. If any cell fails or disconnect
does not happen, choose **Runtime → Disconnect and delete runtime**.

Sources: [model card](https://huggingface.co/Qwen/Qwen3.5-2B),
[pinned Transformers implementation](https://github.com/huggingface/transformers/blob/v5.18.0/src/transformers/models/qwen3_5/modeling_qwen3_5.py).
**Prepared, not run on Colab yet.**


## 1. Budget, Drive, and save location

This starts the clock and a watchdog before installation. The entered rate determines
when to stop; Drive saves the report even if the model compatibility check fails.


In [ ]:
import sys, time, json, math, threading, subprocess, datetime, uuid
from pathlib import Path
from google.colab import drive, runtime

SESSION_START = time.monotonic()
SMOKE_CAP = 5.0
RESERVE = 20.0
BALANCE = float(input("Current Colab compute units available (about 50): "))
RATE = float(input("LIVE usage rate in compute units per hour, from Colab Resources: "))
if not math.isfinite(BALANCE) or not math.isfinite(RATE) or RATE <= 0 or BALANCE <= RESERVE:
    raise ValueError("Enter a finite positive rate and balance above the 20-unit reserve.")
SMOKE_CAP = min(SMOKE_CAP, BALANCE - RESERVE)
drive.mount('/content/drive')
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
OUT = Path('/content/drive/MyDrive/stop-overthinking/rpmr-smoke') / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
REPORT = dict(status='started', run_id=RUN_ID, model='Qwen/Qwen3.5-2B',
              initial_balance=BALANCE, entered_units_per_hour=RATE,
              estimated_unit_cap=SMOKE_CAP, reserve_units=RESERVE, checks=[],
              scope='numerical and cache compatibility only; not loop recovery or accuracy')
LOCK = threading.RLock()
STOP_WATCHDOG = threading.Event()

def used_units():
    return (time.monotonic() - SESSION_START) / 3600 * RATE

def save_report():
    with LOCK:
        REPORT['wall_seconds'] = round(time.monotonic() - SESSION_START, 3)
        REPORT['estimated_units_used'] = round(used_units(), 4)
        temporary = OUT / 'report.json.tmp'
        temporary.write_text(json.dumps(REPORT, indent=2))
        temporary.replace(OUT / 'report.json')

def guard():
    if used_units() >= max(0, SMOKE_CAP - 0.1):
        raise TimeoutError('Estimated smoke-unit ceiling reached. Stop this session.')

def watchdog():
    while not STOP_WATCHDOG.wait(1):
        if used_units() >= max(0, SMOKE_CAP - 0.1):
            with LOCK:
                REPORT['status'] = 'budget_stopped'
                save_report()
            print('Budget ceiling: disconnecting runtime. Report saved to Drive.', flush=True)
            try:
                runtime.unassign()
            except Exception as exc:
                with LOCK:
                    REPORT['disconnect_error'] = str(exc)
                    save_report()
                print('Disconnect manually: Runtime → Disconnect and delete runtime.', flush=True)
            return

save_report()
threading.Thread(target=watchdog, daemon=True).start()
print(f'Smoke cap {SMOKE_CAP:.2f} units; reserve {RESERVE:.0f}. Results: {OUT}')
guard()


## 2. The new calculation

This defines PyTorch calculations for the GPU run. The protected span uses selected
query vectors; their immediate memory readouts stay unchanged. Precision-aware rank
thresholds discard rounding noise. This does not prove preservation of task meaning.


In [ ]:
def orthogonal_basis(torch, columns, reference_scale=None, max_rank=None):
    if columns.shape[1] == 0:
        return columns[:, :0]
    u, singular, _ = torch.linalg.svd(columns, full_matrices=False)
    scale = singular[0] if reference_scale is None else reference_scale
    threshold = max(columns.shape) * torch.finfo(columns.dtype).eps * scale
    rank = int((singular > threshold).sum().item())
    if max_rank is not None:
        rank = min(rank, max_rank)
    return u[:, :rank]

def repair(torch, state, reference, probes, cycle, eta=0.25, cap_fraction=0.05):
    if state.ndim != 2 or state.shape != reference.shape:
        raise ValueError('Current and reference memory must have the same matrix shape.')
    if probes.ndim != 2 or cycle.ndim != 2 or probes.shape[0] != state.shape[0] or cycle.shape[0] != state.shape[0]:
        raise ValueError('Direction rows must match the memory key dimension.')
    if not 0 <= eta <= 1 or not 0 < cap_fraction <= 1:
        raise ValueError('Invalid repair strength or norm cap.')
    s, ref, q, c = [x.detach().float() for x in (state, reference, probes, cycle)]
    if not all(torch.isfinite(x).all().item() for x in (s, ref, q, c)):
        raise ValueError('Non-finite memory or direction.')
    u = orthogonal_basis(torch, q)
    scale = torch.linalg.matrix_norm(c, ord=2) if c.shape[1] else 0.0
    v = orthogonal_basis(torch, c - u @ (u.T @ c), reference_scale=scale)
    drift = v @ (v.T @ (s - ref))
    proposed_norm = eta * torch.linalg.norm(drift).item()
    allowed_norm = cap_fraction * torch.linalg.norm(s).item()
    effective_eta = eta * min(1.0, allowed_norm / max(proposed_norm, 1e-30))
    edited = s - effective_eta * drift
    error = torch.linalg.norm(u.T @ (edited - s)).item()
    return edited, u, v, dict(protected_rank=u.shape[1], eligible_rank=v.shape[1],
                              effective_eta=effective_eta, immediate_readout_error=error,
                              edit_norm=torch.linalg.norm(edited-s).item())


## 3. Run the Colab smoke check, save, and disconnect

Installs the pinned Transformers version, checks GPU calculations, and loads the
official 2B model without LoRA. Captures actual prompt queries from one DeltaNet
layer, checks a real recurrent-cache edit, and records 16-token decoding speed.
The zero-strength continuation must match the unchanged-cache continuation.
Full-attention and convolution caches are copied along with the recurrent states.
Nothing here establishes better answers or detects a reasoning loop.


In [ ]:
try:
    guard()
    remaining_seconds = max(1, (SMOKE_CAP - used_units() - 0.1) / RATE * 3600)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformers==5.18.0', 'accelerate', 'safetensors'],
                   check=True, timeout=remaining_seconds)
    guard()
    import copy, inspect, hashlib, importlib.metadata
    import torch
    import transformers
    from transformers import AutoTokenizer
    from huggingface_hub import HfApi
    from transformers.models.qwen3_5 import modeling_qwen3_5 as implementation
    if not torch.cuda.is_available():
        raise RuntimeError('Choose a Colab GPU runtime before running this notebook.')
    torch.manual_seed(3407)
    torch.cuda.manual_seed_all(3407)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    REPORT['environment'] = dict(python=sys.version, torch=torch.__version__,
                                 transformers=transformers.__version__,
                                 gpu=torch.cuda.get_device_name(0), model_dtype=str(dtype),
                                 memory_math_dtype='torch.float32')
    save_report()

    # Actual model-head dimensions, on this Colab GPU.
    for seed in range(4):
        guard()
        generator = torch.Generator(device='cuda').manual_seed(3407 + seed)
        s, ref = torch.randn(2, 128, 128, device='cuda', generator=generator)
        q = torch.randn(128, 8, device='cuda', generator=generator)
        c = torch.randn(128, 4, device='cuda', generator=generator)
        edited, u, v, info = repair(torch, s, ref, q, c)
        torch.testing.assert_close(u.T @ edited, u.T @ s, atol=3e-5, rtol=3e-5)
        torch.testing.assert_close(v.T @ (edited-ref), (1-info['effective_eta']) * v.T @ (s-ref), atol=3e-5, rtol=3e-5)
        noop, _, eligible, _ = repair(torch, s, ref, q, q[:, :4])
        assert eligible.shape[1] == 0, 'Projection noise became an eligible repair direction.'
        torch.testing.assert_close(noop, s, atol=0, rtol=0)
    REPORT['checks'].append('GPU projection, contraction and protected-overlap checks passed')
    save_report()
    guard()

    MODEL_ID = 'Qwen/Qwen3.5-2B'
    revision = HfApi().model_info(MODEL_ID).sha
    REPORT['model_revision'] = revision
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=revision)
    loader = getattr(transformers, 'AutoModelForMultimodalLM', None)
    if loader is None:
        loader = transformers.AutoModelForImageTextToText
    model = loader.from_pretrained(MODEL_ID, revision=revision, dtype=dtype,
                                    device_map={'': 0}, attn_implementation='sdpa')
    model.eval()
    selected = next(m for m in model.modules() if m.__class__.__name__ == 'Qwen3_5GatedDeltaNet')
    layer_index = selected.layer_idx
    REPORT['selected_layer'] = layer_index
    REPORT['implementation_sha256'] = hashlib.sha256(Path(inspect.getfile(implementation)).read_bytes()).hexdigest()
    guard()

    question = 'Write a Python function that returns the sum of even integers in a list. Briefly plan your approach.'
    prompt = tokenizer.apply_chat_template([dict(role='user', content=question)],
                                           tokenize=False, add_generation_prompt=True, enable_thinking=True)
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to('cuda')
    # Capture actual post-convolution queries at the selected layer's rule call.
    captured = dict(active=False, probes=None)
    def start_capture(module, args):
        captured['active'] = True
    def end_capture(module, args, output):
        captured['active'] = False
    handles = [selected.register_forward_pre_hook(start_capture), selected.register_forward_hook(end_capture)]
    originals = {}
    def wrap_rule(original):
        def wrapped(*args, **kwargs):
            if captured['active'] and captured['probes'] is None:
                query = args[0] if args else kwargs['query']
                vectors = torch.nn.functional.normalize(query[0, :, 0, :].detach().float(), dim=-1)
                captured['probes'] = orthogonal_basis(torch, vectors.T, max_rank=8)
            return original(*args, **kwargs)
        return wrapped
    for name in ('torch_chunk_gated_delta_rule', 'torch_recurrent_gated_delta_rule'):
        originals[name] = getattr(implementation, name)
        setattr(implementation, name, wrap_rule(originals[name]))
    try:
        with torch.no_grad():
            output = model(input_ids=input_ids, attention_mask=torch.ones_like(input_ids),
                           use_cache=True, logits_to_keep=1)
    finally:
        for handle in handles:
            handle.remove()
        for name, original in originals.items():
            setattr(implementation, name, original)
    if captured['probes'] is None:
        raise RuntimeError('Active kernel bypassed query capture. Cache integration is not validated.')
    cache = output.past_key_values
    memory = cache.layers[layer_index].recurrent_states[0]
    assert memory.ndim == 4 and tuple(memory.shape[-2:]) == (128, 128)
    assert memory.dtype == torch.float32, 'Cache precision changed; validate before editing.'
    reference = memory[0, 0].detach().clone()
    cursor = output.logits[:, -1].argmax(dim=-1, keepdim=True)
    generated = []
    torch.cuda.synchronize()
    decode_start = time.monotonic()
    with torch.no_grad():
        for step in range(16):
            guard()
            generated.append(int(cursor.item()))
            output = model(input_ids=cursor,
                           attention_mask=torch.ones((1, input_ids.shape[1]+step+1), device='cuda', dtype=torch.long),
                           past_key_values=cache, use_cache=True, logits_to_keep=1)
            cache = output.past_key_values
            cursor = output.logits[:, -1].argmax(dim=-1, keepdim=True)
    torch.cuda.synchronize()
    seconds = time.monotonic() - decode_start
    REPORT['decode'] = dict(tokens=16, seconds=seconds, tokens_per_second=16/seconds,
                            mode='greedy diagnostic', includes_online_repair=False,
                            raw_prefix=tokenizer.decode(generated, skip_special_tokens=False))
    current = cache.layers[layer_index].recurrent_states[0][0, 0].detach().clone()
    drift = current - reference
    cycle = orthogonal_basis(torch, drift, max_rank=4)
    edited, u, v, info = repair(torch, current, reference, captured['probes'], cycle)
    torch.testing.assert_close(u.T @ edited, u.T @ current, atol=3e-5, rtol=3e-5)
    if info['edit_norm'] == 0:
        raise RuntimeError('No nonzero eligible cache edit; compatibility check is incomplete.')
    baseline_cache = copy.deepcopy(cache)
    zero_cache = copy.deepcopy(cache)
    repair_cache = copy.deepcopy(cache)
    zero_edit, _, _, _ = repair(torch, current, reference, captured['probes'], cycle, eta=0)
    zero_cache.layers[layer_index].recurrent_states[0][0, 0].copy_(zero_edit)
    repair_cache.layers[layer_index].recurrent_states[0][0, 0].copy_(edited)
    torch.testing.assert_close(cache.layers[layer_index].recurrent_states[0][0, 0], current, atol=0, rtol=0)
    branch_logits = {}
    with torch.no_grad():
        for label, branch in [('unchanged', baseline_cache), ('zero_strength', zero_cache), ('forced_repair', repair_cache)]:
            guard()
            branch_output = model(input_ids=cursor,
                                  attention_mask=torch.ones((1, input_ids.shape[1]+17), device='cuda', dtype=torch.long),
                                  past_key_values=branch, use_cache=True, logits_to_keep=1)
            assert torch.isfinite(branch_output.logits).all().item()
            branch_logits[label] = branch_output.logits.detach().float()
    torch.testing.assert_close(branch_logits['unchanged'], branch_logits['zero_strength'], atol=0, rtol=0)
    info['state_shape'] = list(cache.layers[layer_index].recurrent_states[0].shape)
    info['max_next_logit_difference'] = (branch_logits['forced_repair']-branch_logits['unchanged']).abs().max().item()
    info['forced_edit_not_loop_detection'] = True
    REPORT['cache_repair'] = info
    REPORT['checks'].append('Actual prompt query capture, nonzero cache edit and finite continuation passed')
    REPORT['checks'].append('Zero-strength cache continuation exactly matches unchanged continuation')
    REPORT['peak_gpu_memory_gb'] = torch.cuda.max_memory_allocated()/1e9
    guard()
    REPORT['status'] = 'passed'
except Exception as exc:
    with LOCK:
        if REPORT.get('status') != 'budget_stopped':
            REPORT['status'] = 'failed'
        REPORT['error'] = f'{type(exc).__name__}: {exc}'
    import traceback
    traceback.print_exc()
finally:
    save_report()
    print(json.dumps(REPORT, indent=2))
    print('Report saved to', OUT / 'report.json')
    try:
        from google.colab import files
        files.download(str(OUT / 'report.json'))
    except Exception as exc:
        print('Download unavailable; report remains on Drive:', exc)
    try:
        runtime.unassign()
    except Exception as exc:
        REPORT['disconnect_error'] = str(exc)
        save_report()
        print('Disconnect manually: Runtime → Disconnect and delete runtime.')
    else:
        STOP_WATCHDOG.set()
